# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [16]:
# Install DuckDB
!pip -q install duckdb huggingface_hub

In [17]:
import duckdb
import getpass

hf_token = getpass.getpass("Enter your Hugging Face READ token (hf_...): ")

# Save the Hugging Face login for this Colab session
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)

print("Hugging Face login successful")

Enter your Hugging Face READ token (hf_...): ··········
Hugging Face login successful


In [18]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute("DROP SECRET IF EXISTS hf_token")

con.execute("""
CREATE SECRET hf_token (
    TYPE huggingface,
    PROVIDER credential_chain
)
""")

print("DuckDB + Hugging Face connected")

DuckDB + Hugging Face connected


In [19]:
# FlyRank warehouse

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"
CLI = f"{REL}/dim_clients.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("Warehouse paths ready")
print("Feature window: February 2026")
print("Label window: March 2026")

Warehouse paths ready
Feature window: February 2026
Label window: March 2026


In [20]:
test = con.execute(f"""
    SELECT *
    FROM {FEB}
    LIMIT 5
""").fetchdf()

display(test)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02
1,2026-02-01,client_e547b89c05043229,content_1eea820697c3b95a,True,True,True,False,13,0,85,...,0,0,0,0,0,0,0,0,0,2026-02
2,2026-02-01,client_e547b89c05043229,content_ccbb253f142217c3,True,True,True,True,59,0,1001,...,0,0,0,0,0,0,0,0,0,2026-02
3,2026-02-01,client_e547b89c05043229,content_ae16a6b9cf64c80a,True,True,True,False,17,0,287,...,0,0,0,0,0,0,0,0,0,2026-02
4,2026-02-01,client_e547b89c05043229,content_acf700633f016e5a,True,True,True,False,6,0,27,...,0,0,0,0,0,0,0,0,0,2026-02


In [22]:
# Show all columns and their data types

schema = con.execute(f"""
    DESCRIBE SELECT *
    FROM {FEB}
""").fetchdf()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [23]:
# Print all 31 column names clearly

for i, col in enumerate(schema["column_name"], start=1):
    print(i, col)

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


In [24]:
# Verification 1: check the row grain

grain_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_grain_rows
    FROM {FEB}
""").fetchdf()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows
0,7355108,7355108


**Unit of analysis:** One row represents the daily performance of one content item for one client on one report date. The grain is defined by `report_date + client_hash_id + content_hash_id`.

**Feature window:** February 2026.

**Label window:** March 2026. March information is used only to construct the label and is not used as an input feature.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [28]:
# Build the February feature frame

feb_features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_feb,
        SUM(gsc_clicks) AS clicks_feb,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_feb,
        SUM(ga4_pageviews) AS pageviews_feb,
        SUM(ga4_sessions) AS sessions_feb
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").fetchdf()

display(feb_features.head())
print("Rows:", len(feb_features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,pageviews_feb,sessions_feb
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,28.886364,6.0,5.0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.273467,13.0,13.0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,8.182346,1.0,1.0
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,8.032653,1.0,1.0
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.581699,0.0,0.0


Rows: 153559


In [29]:
# Build the March label from measured GSC data only

march_label = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_mar,
        SUM(gsc_impressions) AS impressions_mar,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS measured_days_mar
    FROM {MAR}
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) > 0
""").fetchdf()

march_label["went_dark"] = (
    (march_label["clicks_mar"] == 0) &
    (march_label["measured_days_mar"] > 0)
).astype(int)

display(march_label.head())

print("Rows:", len(march_label))
print("Went dark:", int(march_label["went_dark"].sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,clicks_mar,impressions_mar,measured_days_mar,went_dark
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,77.0,24,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,4.0,602.0,29,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,1.0,810.0,29,0
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,0.0,82.0,27,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,6.0,1858.0,30,0


Rows: 176738
Went dark: 107901


In [30]:
# Combine February features with the March outcome label

final_frame = feb_features.merge(
    march_label[
        [
            "client_hash_id",
            "content_hash_id",
            "clicks_mar",
            "impressions_mar",
            "measured_days_mar",
            "went_dark"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

display(final_frame.head())

print("Rows:", len(final_frame))
print("Columns:", list(final_frame.columns))

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,pageviews_feb,sessions_feb,clicks_mar,impressions_mar,measured_days_mar,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,28.886364,6.0,5.0,1.0,768.0,29,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.273467,13.0,13.0,6.0,3071.0,29,0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,8.182346,1.0,1.0,0.0,547.0,29,1
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,8.032653,1.0,1.0,0.0,209.0,29,1
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.581699,0.0,0.0,1.0,500.0,29,0


Rows: 134238
Columns: ['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'avg_position_feb', 'pageviews_feb', 'sessions_feb', 'clicks_mar', 'impressions_mar', 'measured_days_mar', 'went_dark']


In [25]:
# Inspect the main performance fields

sample = con.execute(f"""
    SELECT
        report_date,
        gsc_data_available,
        ga4_data_available,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_sessions,
        ga4_users,
        scroll_events
    FROM {FEB}
    LIMIT 10
""").fetchdf()

display(sample)

,report_date,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,scroll_events
0,2026-02-01,True,False,57,0,31.192982,0,0,0,0
1,2026-02-01,True,False,13,0,6.538462,0,0,0,0
2,2026-02-01,True,True,59,0,16.966102,2,2,2,0
3,2026-02-01,True,False,17,0,16.882353,0,0,0,0
4,2026-02-01,True,False,6,0,4.500000,0,0,0,0
5,2026-02-01,True,True,27,1,6.925926,1,1,1,0
6,2026-02-01,True,False,10,0,24.600000,0,0,0,0
7,2026-02-01,True,False,5,0,18.200000,0,0,0,0
8,2026-02-01,True,False,81,0,31.740741,0,0,0,0
9,2026-02-01,True,False,24,0,3.291667,0,0,0,0


In [31]:
# Deliberate leakage experiment
# clicks_mar is part of the March outcome, so using it as a feature is leakage.

leak_prediction = (final_frame["clicks_mar"] == 0).astype(int)

leak_accuracy = (
    leak_prediction == final_frame["went_dark"]
).mean()

print(f"Leak experiment accuracy: {leak_accuracy:.1%}")

Leak experiment accuracy: 100.0%


In [32]:
# Remove March information so it cannot be used as a model feature

model_frame = final_frame.drop(
    columns=[
        "clicks_mar",
        "impressions_mar",
        "measured_days_mar"
    ]
)

display(model_frame.head())

print("Final columns:")
print(list(model_frame.columns))

,client_hash_id,content_hash_id,impressions_feb,clicks_feb,avg_position_feb,pageviews_feb,sessions_feb,went_dark
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,28.886364,6.0,5.0,0
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.273467,13.0,13.0,0
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,8.182346,1.0,1.0,1
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,8.032653,1.0,1.0,1
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.581699,0.0,0.0,0


Final columns:
['client_hash_id', 'content_hash_id', 'impressions_feb', 'clicks_feb', 'avg_position_feb', 'pageviews_feb', 'sessions_feb', 'went_dark']


### Feature / label / context / excluded

**Features**
- `impressions_feb` — total GSC impressions during February.
- `clicks_feb` — total GSC clicks during February.
- `avg_position_feb` — impression-weighted average GSC position during February.
- `pageviews_feb` — total GA4 pageviews during February.
- `sessions_feb` — total GA4 sessions during February.

**Label**
- `went_dark` — 1 when the content had zero measured GSC clicks during March; otherwise 0.

**Context / identifiers**
- `report_date` — identifies the daily observation date.
- `client_hash_id` — identifies the client.
- `content_hash_id` — identifies the content item.
- `gsc_data_available` — indicates whether GSC measurements are available.
- `ga4_data_available` — indicates whether GA4 measurements are available.

**Excluded**
- March clicks and impressions are excluded from model features because they occur after the February feature window and would leak future information.
- March measurement availability is also excluded from model features because it is used to construct the label.
- Identifiers are retained for joining observations but are not model features.

### Feature availability

| Feature | Available when? |
|---|---|
| `impressions_feb` | GSC data is available during February |
| `clicks_feb` | GSC data is available during February |
| `avg_position_feb` | GSC impressions and position data are available |
| `pageviews_feb` | GA4 data is available during February |
| `sessions_feb` | GA4 data is available during February |

All model features come from the February feature window. March data is used only to construct the label.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Grain check

I claim that one row represents one `content_id`. I verify this by comparing total rows with distinct content IDs.

### Count check

I report the number of observations and the number of unique content items in the analysis slice.

In [26]:
# Verification 2: row count and date span for the feature window

window_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {FEB}
""").fetchdf()

display(window_check)

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


In [27]:
# Verification 3: check GSC data availability

availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS rows_with_gsc_data
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
""").fetchdf()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc_data
0,2621783


### Missing-value check

I check missingness in the fields used by the contract. Missing values are reported rather than silently treated as valid observations.

### Time-window check

I verify the earliest and latest observation dates and report the number of distinct dates represented in the analysis slice.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Data limits

This contract cannot tell us whether a content item was truly unsuccessful for reasons outside the observed search data.

Important limitations:

- The history is limited to the available warehouse time period, so long-term trends may not be observable.
- GSC and GA4 data are not available for every client/content observation.
- Missing GSC measurements cannot automatically be interpreted as zero clicks or zero impressions.
- February and March are adjacent windows, so changes around the boundary may be sensitive to the chosen cutoff.
- The March label only describes the observed March outcome; it does not establish why the outcome happened.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.